# Getting started with MOLA

MOLA characterizes the landscape of a continuous multi-objective optimization problem: sample it, evaluate it, and compute 49 landscape features (paper Table 1) describing its distances, multimodality, evolvability, and ruggedness. This notebook is the **first five minutes** only -- install, run one command, read the result, know where to go next. For what each of the 49 features actually means, see the four per-class notebooks under [`notebooks/`](../../notebooks/) instead; this one doesn't repeat that content.

## Install

```bash
conda env create -f ../../environment.yml
conda activate MOLA
```

See the main [`README.md`](../../README.md) for the `venv` alternative. Everything below assumes this environment is active (it's what this notebook itself was executed with).

## Option A: the CLI, one command

`mola run` samples a jMetalPy problem, evaluates it, and characterizes it in one step -- no intermediate file. It's the most convenient entry point for a shell, a script, or an AI agent that just wants a structured result back.

In [1]:
!mola run ZDT1 --variables 5 --sample-size 200 --seed 42

sample_size: 200
num_obj: 2
num_var: 5
f_cor: -0.383048
dist_x_avg: 0.452827
dist_x_max: 1.78542
dist_f_avg: 0.25507
dist_f_max: 5.97386
nd_n: 0.055
supp_n: 0.454545
hv: 5.33762
dist_x_nd_avg: 0.374214
dist_x_nd_max: 1.19961
fdc: 0.399784
rank_avg: 6.92
rank_max: 19
rank_ent: 4.06004
slo_n: 0.065
slo_dist_avg: 0.434536
slo_dist_max: 1.35982
plo_n: 0.42
plo_dist_avg: 0.456864
plo_dist_max: 1.78542
nd_per_plo: 0.130952
length_aws: 0.866667
eval_aws: 7.3
sup_avg_neig: 0.172
inf_avg_neig: 0.164
inc_avg_neig: 0.664
lnd_avg_neig: 0.532
lsupp_avg_neig: 0.449
dist_x_avg_neig: 0.341862
dist_f_avg_neig: 0.56391
dist_f_dist_x_avg_neig: 1.64953
diff_f_avg_neig: 0.328872
diff_f_dist_x_avg_neig: 0.962002
hv_avg_neig: 1.52858
hvd_avg_neig: 0.467182
nhv_avg_neig: 2.45858
dist_x_cor_neig: 0.371807
dist_f_cor_neig: 0.322272
sup_cor_neig: 0.134714
inf_cor_neig: 0.114504
inc_cor_neig: 0.27992
lnd_cor_neig: 0.178363
lsupp_cor_neig: 0.129372
dist_f_dist_x_cor_neig: 0.284597
diff_f_cor_neig: 0.322526
diff_f_

Every `mola` command is documented in depth via its own `--help`:

In [2]:
!mola run --help

                                                                                
 Usage: mola run [OPTIONS] {problem}                                            
                                                                                
 Sample, evaluate, and characterize a jMetalPy problem in one step -- no file   
 needed.                                                                        
                                                                                
 The most convenient entry point: straight from a jMetalPy problem name to its  
 49 landscape                                                                   
 features plus sample_size/num_obj/num_var, with no intermediate interchange    
 file. Well suited                                                              
 to scripted or AI-agent use, where a single command producing a structured     
 result matters more                                                            
 than inspecting the interme

## Option B: the Python API

The CLI is a thin wrapper around two functions: an **adapter** that samples and evaluates a problem (`mola.adapters.jmetalpy.sample_problem`, one per framework), and the framework-independent **core** that computes every feature from the result (`mola.characterize.characterize`). Calling them directly is the same workflow, in-process:

In [3]:
from jmetal.problem import ZDT1

from mola.adapters.jmetalpy import sample_problem
from mola.characterize import characterize

problem = ZDT1(number_of_variables=5)
sample = sample_problem(problem, sample_size=200, seed=42)
result = characterize(sample)

print(f"{sample.problem}: {sample.size} solutions")
print(f"{sample.number_of_variables} variables, {sample.number_of_objectives} objectives")

ZDT1: 200 solutions
5 variables, 2 objectives


## Reading a result

`result` is a plain `dict[str, float]`: 3 always-reported metadata fields (`sample_size`, `num_obj`, `num_var`), then the 49 features in the paper's own class order -- global, multimodality, evolvability, ruggedness.

In [4]:
print(f"{len(result)} keys total\n")
for name in ["sample_size", "nd_n", "hv", "slo_n", "sup_avg_neig", "dist_x_cor_neig"]:
    print(f"{name:16s} {result[name]:.4f}")

52 keys total

sample_size      200.0000
nd_n             0.0550
hv               5.3376
slo_n            0.0650
sup_avg_neig     0.1720
dist_x_cor_neig  0.3718


## Working from a file instead

The core never sees a problem object -- only the *interchange* file format (a CSV plus a sidecar metadata JSON, [`examples/sample.csv`](sample.csv) / [`examples/sample.json`](sample.json) is a small checked-in example). This is what lets `mola characterize` work on a sample written by *any* adapter, including a future Java one -- not just jMetalPy's.

In [5]:
!mola characterize sample.csv

sample_size: 20
num_obj: 2
num_var: 3
f_cor: -0.190977
dist_x_avg: 0.428467
dist_x_max: 1.41292
dist_f_avg: 0.293865
dist_f_max: 7.37343
nd_n: 0.15
supp_n: 1
hv: 5.77966
dist_x_nd_avg: 0.468702
dist_x_nd_max: 1.04654
fdc: 1
rank_avg: 2.7
rank_max: 6
rank_ent: 2.70869
slo_n: 0.225
slo_dist_avg: 0.415347
slo_dist_max: 0.972261
plo_n: 0.55
plo_dist_avg: 0.454776
plo_dist_max: 1.39658
nd_per_plo: 0.272727
length_aws: 0.65
eval_aws: 4.4
sup_avg_neig: 0.166667
inf_avg_neig: 0.2
inc_avg_neig: 0.633333
lnd_avg_neig: 0.5
lsupp_avg_neig: 0.5
dist_x_avg_neig: 0.348451
dist_f_avg_neig: 1.03771
dist_f_dist_x_avg_neig: 2.97805
diff_f_avg_neig: 0.587299
diff_f_dist_x_avg_neig: 1.68546
hv_avg_neig: 1.60052
hvd_avg_neig: 0.789413
nhv_avg_neig: 2.57995
dist_x_cor_neig: 0.0924853
dist_f_cor_neig: 0.296521
sup_cor_neig: -0.116589
inf_cor_neig: -0.0473763
inc_cor_neig: -0.100596
lnd_cor_neig: 0.208853
lsupp_cor_neig: 0.208853
dist_f_dist_x_cor_neig: 0.436169
diff_f_cor_neig: 0.284146
diff_f_dist_x_cor_neig

## Where to go next

- [`notebooks/`](../../notebooks/) -- one notebook per feature class (global, multimodality, evolvability, ruggedness), each feature's definition paired with a real, executed example.
- [`examples/getting_started/quickstart.py`](quickstart.py) -- this same Python-API workflow as a plain, directly-runnable script.
- [`CLAUDE.md`](../../CLAUDE.md) -- the full design brief: the interchange schema, every design decision, and the complete 49-feature implementation matrix.
- [`llms.txt`](../../llms.txt) -- a short, link-heavy summary meant for an AI agent reading this repository for the first time.